In [3]:
import nflreadpy
import pandas as pd
import numpy as np
years = list(range(2016, 2026))

ModuleNotFoundError: No module named 'nflreadpy'

In [ ]:
def add_games_played_column(weekly_data):
    # Filter the data to include only regular season games
    weekly_data = weekly_data[weekly_data['season_type'] == 'REG']
    
    # Calculate games played per player per season
    weekly_data['GP'] = weekly_data.groupby(['player_display_name', 'season'])['week'].transform('count')
    
    return weekly_data

def add_epa_averages(weekly_data):
    # Calculate the averages of rushing and receiving EPA per player per season
    epa_averages = weekly_data.groupby(['player_display_name', 'season']).agg({
        'rushing_epa': 'mean',
        'receiving_epa': 'mean'
    }).reset_index()
    
    # Rename the columns to indicate they are averages
    epa_averages.rename(columns={
        'rushing_epa': 'avg_rushing_epa',
        'receiving_epa': 'avg_receiving_epa'
    }, inplace=True)
    
    # Merge the averages back into the original weekly_data DataFrame
    weekly_data = pd.merge(weekly_data, epa_averages, on=['player_display_name', 'season'], how='left')
    
    return weekly_data


# nflreadpy weekly uses 'team' instead of 'recent_team' — rename for compatibility
data = nflreadpy.load_player_stats(years, summary_level='week').to_pandas()
data = data.rename(columns={'team': 'recent_team'})
weekly_data = data.sort_values(by=['player_display_name', 'week'], ascending=[True, True])
weekly_data = add_games_played_column(weekly_data)
weekly_data = add_epa_averages(weekly_data)


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/1212993133.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weekly_data['GP'] = weekly_data.groupby(['player_display_name', 'season'])['week'].transform('count')


In [ ]:
import pandas as pd
from datetime import datetime

def merge_recent_team(seasonal_data, weekly_data):
    # Group weekly data to get the most recent team and other first-occurring attributes
    grouped_weekly = weekly_data.groupby(["player_id", "season"]).agg({
        "GP": "first",  # Summing up games played if weekly data has entries per game
    }).reset_index()
    
    # Merge grouped weekly data with seasonal data
    merged_data = pd.merge(grouped_weekly, seasonal_data, on=["player_id", "season"], how="left")
    return merged_data

# nflreadpy uses "passing_interceptions"/"sacks_suffered" -- rename to match downstream column selections
season_data = (nflreadpy.load_player_stats(years, summary_level="reg").to_pandas()
               .rename(columns={"passing_interceptions": "interceptions", "sacks_suffered": "sacks"}))

# Merge and calculate additional columns
season_data = merge_recent_team(season_data, weekly_data)

# nflreadpy rosters do not include "age" -- compute from birth_date relative to Sep 1 of that season
season_rosters = (nflreadpy.load_rosters(years).to_pandas()
                  .rename(columns={"gsis_id": "player_id", "full_name": "player_name"}))
season_rosters["age"] = season_rosters.apply(
    lambda row: (datetime(int(row["season"]), 9, 1) - pd.Timestamp(row["birth_date"])).days // 365
    if pd.notna(row["birth_date"]) else None, axis=1
)
season_rosters = season_rosters[["season","team","position","player_name","player_id","age","status"]]
season_data = pd.merge(season_data, season_rosters[["season","player_id","age","status"]], on=["season","player_id"], how="left")

season_data


,player_id,season,GP,player_name,player_display_name,position,position_group,headshot_url,season_type,recent_team,...,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance_list,fantasy_points,fantasy_points_ppr,age,status
0,00-0004091,2016,16.0,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,REG,SF,...,0.970588,0,0,0,0,None,0.0,0.0,41.0,ACT
1,00-0004091,2017,16.0,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,REG,ARI,...,0.884615,1,1,0,0,30,0.0,0.0,42.0,ACT
2,00-0004091,2018,10.0,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,REG,ARI,...,1.000000,0,0,0,0,None,0.0,0.0,43.0,RES
3,00-0016919,2016,16.0,A.Vinatieri,Adam Vinatieri,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,REG,IND,...,1.000000,0,0,0,0,None,0.0,0.0,43.0,ACT
4,00-0016919,2017,15.0,A.Vinatieri,Adam Vinatieri,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,REG,IND,...,0.916667,1,1,0,0,51,0.0,0.0,44.0,ACT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19517,00-0040751,2025,11.0,W.Johnson,Will Johnson,CB,DB,https://static.www.nfl.com/image/upload/f_auto...,REG,ARI,...,NaN,0,0,0,0,None,0.0,0.0,22.0,ACT
19518,00-0040752,2025,6.0,O.Oladejo,Oluwafemi Oladejo,LB,LB,https://static.www.nfl.com/image/upload/f_auto...,REG,TEN,...,NaN,0,0,0,0,None,0.0,0.0,21.0,RES
19519,00-0040756,2025,6.0,S.Stewart,Shemar Stewart,DE,DL,https://static.www.nfl.com/image/upload/f_auto...,REG,CIN,...,NaN,0,0,0,0,None,0.0,0.0,21.0,ACT
19520,00-0040782,2025,15.0,I.Bond,Isaiah Bond,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,REG,CLE,...,NaN,0,0,0,0,None,36.7,54.7,21.0,ACT


In [ ]:
#QB_NGS
passing_ngs = nflreadpy.load_nextgen_stats(years, stat_type='passing').to_pandas()
passing_ngs = passing_ngs.drop(['player_gsis_id','player_first_name','player_last_name','player_jersey_number','player_short_name','week'], axis=1, errors='ignore')
passing_ngs = passing_ngs[passing_ngs['season_type'] == 'REG']
passing_ngs = passing_ngs.groupby(['season', 'season_type','player_display_name', 'player_position','team_abbr']).mean(numeric_only=True)
passing_ngs = passing_ngs.reset_index()
keep_columns = ['season', 'season_type', 'player_display_name', 'player_position', 'team_abbr']
new_columns = {col: col + " / game" if col not in keep_columns else col for col in passing_ngs.columns}
passing_ngs.rename(columns=new_columns, inplace=True)
qb_data = season_data[season_data['position'] == "QB"]
# qb_data = pd.merge(passing_ngs, qb_data, how='right', on=['player_display_name', 'season', 'season_type'])

# Note: QBR (import_qbr) is not available in nflreadpy — QBR merge skipped
qb_data['comp %'] = qb_data['completions'] / qb_data['attempts']
qb_data['td:int'] = qb_data['passing_tds'] / qb_data['interceptions']
qb_data['yards/attempts'] = qb_data['passing_yards'] / qb_data['attempts']
qb_data['yards/comp'] = qb_data['passing_yards'] / qb_data['completions']
qb_data['yards/carry'] = qb_data['rushing_yards'] / qb_data['carries']
qb_data["passer rating"] = ((((qb_data['comp %'] - 0.3) * 5) + (((qb_data['passing_yards']/qb_data['attempts']) - 3) * 0.25) + ((qb_data['passing_tds'] / qb_data['attempts']) * 20) + 2.375 - ((qb_data['interceptions'] / qb_data['attempts']) * 25)) / 6) * 100


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/695836654.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qb_data['comp %'] = qb_data['completions'] / qb_data['attempts']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/695836654.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qb_data['td:int'] = qb_data['passing_tds'] / qb_data['interceptions']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/695836654.py:16: SettingWithCopyWarning: 
A value is try

In [ ]:
#FLEX_NGS
rushing_ngs = nflreadpy.load_nextgen_stats(years, stat_type='rushing').to_pandas()
rushing_ngs = rushing_ngs.drop(['player_gsis_id','player_first_name','player_last_name','player_jersey_number','player_short_name','week'], axis=1, errors='ignore')
rushing_ngs = rushing_ngs[rushing_ngs['season_type'] == 'REG']
rushing_ngs = rushing_ngs.groupby(['season', 'season_type','player_display_name', 'player_position','team_abbr']).mean()
rushing_ngs = rushing_ngs.reset_index()
keep_columns = ['season', 'season_type', 'player_display_name', 'player_position', 'team_abbr']
new_columns = {col: col + " / game" if col not in keep_columns else col for col in rushing_ngs.columns}
rushing_ngs.rename(columns=new_columns, inplace=True)

rb_data = season_data[season_data['position'].isin(['RB','HB','FB'])]
rb_data['position'] = rb_data['position'].replace({'HB': 'RB', 'FB': 'RB'})
# rb_data = pd.merge(rushing_ngs, rb_data, how='right', on=['player_display_name', 'season', 'season_type'])
rb_data['y/c'] = rb_data['rushing_yards'] / rb_data['carries']
rb_data['y/g'] = rb_data['rushing_yards'] / rb_data['GP']
rb_data['c/g'] = rb_data['carries'] / rb_data['GP']
rb_data['y/rec'] = rb_data['receiving_yards'] / rb_data['receptions']
rb_data['rec/g'] = rb_data['receptions'] / rb_data['GP']
rb_data['y/tgt'] = rb_data['receiving_yards'] / rb_data['targets']
rb_data['catch %'] = 100 * (rb_data['receptions'] / rb_data['targets'])
rb_data['touches'] = rb_data['carries'] + rb_data['receptions']
rb_data['y/touch'] = (rb_data['rushing_yards'] + rb_data['receiving_yards']) / rb_data['touches']
rb_data['rrtd'] = rb_data['rushing_tds'] + rb_data['receiving_tds']
# rb_data.to_pickle("progress position group data/rb_data.pkl")
rb_data


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/2253031923.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rb_data['position'] = rb_data['position'].replace({'HB': 'RB', 'FB': 'RB'})
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/2253031923.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rb_data['y/c'] = rb_data['rushing_yards'] / rb_data['carries']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/2253031923.py:15: SettingWithCopyWarning: 
A value

,player_id,season,GP,player_name,player_display_name,position,position_group,headshot_url,season_type,recent_team,...,y/c,y/g,c/g,y/rec,rec/g,y/tgt,catch %,touches,y/touch,rrtd
120,00-0022999,2016,13.0,J.Kuhn,John Kuhn,RB,RB,https://static.www.nfl.com/image/private/f_aut...,REG,NO,...,2.055556,2.846154,1.384615,4.375000,1.230769,3.500000,80.000000,34,3.147059,5
121,00-0022999,2017,1.0,J.Kuhn,John Kuhn,RB,RB,https://static.www.nfl.com/image/private/f_aut...,REG,NO,...,2.000000,2.000000,1.000000,NaN,0.000000,NaN,NaN,1,2.000000,0
183,00-0023500,2016,16.0,F.Gore,Frank Gore,RB,RB,https://static.www.nfl.com/image/private/f_aut...,REG,IND,...,3.897338,64.062500,16.437500,7.289474,2.375000,5.893617,80.851064,301,4.325581,8
184,00-0023500,2017,16.0,F.Gore,Frank Gore,RB,RB,https://static.www.nfl.com/image/private/f_aut...,REG,IND,...,3.681992,60.062500,16.312500,8.448276,1.812500,6.447368,76.315789,290,4.158621,4
185,00-0023500,2018,14.0,F.Gore,Frank Gore,RB,RB,https://static.www.nfl.com/image/private/f_aut...,REG,MIA,...,4.628205,51.571429,11.142857,10.333333,0.857143,7.750000,75.000000,168,5.035714,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19484,00-0040715,2025,8.0,C.Skattebo,Cam Skattebo,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,REG,NYG,...,4.059406,51.250000,12.625000,8.625000,3.000000,6.468750,75.000000,125,4.936000,7
19487,00-0040719,2025,15.0,B.Tuten,Bhayshul Tuten,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,REG,JAX,...,3.698795,20.466667,5.533333,7.900000,0.666667,5.642857,71.428571,93,4.150538,7
19498,00-0040730,2025,17.0,R.Harvey,RJ Harvey,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,REG,DEN,...,3.698630,31.764706,8.588235,7.574468,2.764706,6.137931,81.034483,193,4.642487,12
19501,00-0040734,2025,17.0,T.Henderson,TreVeyon Henderson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,REG,NE,...,5.061111,53.588235,10.588235,6.314286,2.058824,5.261905,83.333333,215,5.265116,10


In [ ]:
receiving_ngs = nflreadpy.load_nextgen_stats(years, stat_type='receiving').to_pandas()
receiving_ngs = receiving_ngs.drop(['player_gsis_id','player_first_name','player_last_name','player_jersey_number','player_short_name','week'], axis=1, errors='ignore')
receiving_ngs = receiving_ngs[receiving_ngs['season_type'] == 'REG']
receiving_ngs = receiving_ngs.groupby(['season', 'season_type','player_display_name', 'player_position','team_abbr']).mean()
receiving_ngs = receiving_ngs.reset_index()
keep_columns = ['season', 'season_type', 'player_display_name', 'player_position', 'team_abbr']
new_columns = {col: col + " / game" if col not in keep_columns else col for col in receiving_ngs.columns}
receiving_ngs.rename(columns=new_columns, inplace=True)

wrte_data = season_data[season_data['position'].isin(['WR','TE'])]
# wrte_data = pd.merge(receiving_ngs, wrte_data, how='right', on=['player_display_name', 'season', 'season_type'])
wrte_data['y/c'] = wrte_data['rushing_yards'] / wrte_data['carries']
wrte_data['y/g'] = wrte_data['rushing_yards'] / wrte_data['GP']
wrte_data['c/g'] = wrte_data['carries'] / wrte_data['GP']
wrte_data['y/rec'] = wrte_data['receiving_yards'] / wrte_data['receptions']
wrte_data['rec/g'] = wrte_data['receptions'] / wrte_data['GP']
wrte_data['y/tgt'] = wrte_data['receiving_yards'] / wrte_data['targets']
wrte_data['catch %'] = 100 * (wrte_data['receptions'] / wrte_data['targets'])
wrte_data['touches'] = wrte_data['carries'] + wrte_data['receptions']
wrte_data['y/touch'] = (wrte_data['rushing_yards'] + wrte_data['receiving_yards']) / wrte_data['touches']
wrte_data['rrtd'] = wrte_data['rushing_tds'] + wrte_data['receiving_tds']
# wrte_data.to_pickle("progress position group data/wrte_data.pkl")
wrte_data


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/140017211.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wrte_data['y/c'] = wrte_data['rushing_yards'] / wrte_data['carries']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/140017211.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wrte_data['y/g'] = wrte_data['rushing_yards'] / wrte_data['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_22095/140017211.py:14: SettingWithCopyWarning: 
A value is tryin

,player_id,season,GP,player_name,player_display_name,position,position_group,headshot_url,season_type,recent_team,...,y/c,y/g,c/g,y/rec,rec/g,y/tgt,catch %,touches,y/touch,rrtd
18,00-0020337,2016,14.0,S.Smith,Steve Smith,WR,WR,https://static.www.nfl.com/image/private/f_aut...,REG,BAL,...,NaN,0.000000,0.000000,11.414286,5.000000,7.910891,69.306931,70,11.414286,5
42,00-0021547,2016,13.0,A.Gates,Antonio Gates,TE,TE,https://static.www.nfl.com/image/private/f_aut...,REG,LAC,...,NaN,0.000000,0.000000,10.339623,4.076923,5.892473,56.989247,53,10.339623,7
43,00-0021547,2017,16.0,A.Gates,Antonio Gates,TE,TE,https://static.www.nfl.com/image/private/f_aut...,REG,LAC,...,NaN,0.000000,0.000000,10.533333,1.875000,6.076923,57.692308,30,10.533333,3
44,00-0021547,2018,16.0,A.Gates,Antonio Gates,TE,TE,https://static.www.nfl.com/image/private/f_aut...,REG,LAC,...,NaN,0.000000,0.000000,11.892857,1.750000,7.400000,62.222222,28,11.892857,2
48,00-0022044,2016,6.0,A.Johnson,Andre Johnson,WR,WR,https://static.www.nfl.com/image/private/f_aut...,REG,TEN,...,NaN,0.000000,0.000000,9.444444,1.500000,3.695652,39.130435,9,9.444444,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19502,00-0040735,2025,15.0,L.Burden,Luther Burden III,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,REG,CHI,...,6.166667,2.466667,0.400000,13.872340,3.133333,10.866667,78.333333,53,13.000000,2
19503,00-0040736,2025,13.0,M.Taylor,Mason Taylor,TE,TE,https://static.www.nfl.com/image/upload/f_auto...,REG,NYJ,...,NaN,0.000000,0.000000,8.386364,3.384615,5.676923,67.692308,44,8.386364,1
19504,00-0040737,2025,11.0,T.Ferguson,Terrance Ferguson,TE,TE,https://static.www.nfl.com/image/upload/f_auto...,REG,LA,...,0.000000,0.000000,0.090909,21.000000,1.000000,9.240000,44.000000,12,19.250000,3
19506,00-0040739,2025,13.0,E.Arroyo,Elijah Arroyo,TE,TE,https://static.www.nfl.com/image/upload/f_auto...,REG,SEA,...,NaN,0.000000,0.000000,11.933333,1.153846,6.884615,57.692308,15,11.933333,1


In [ ]:
import pandas as pd
AVgrades = pd.read_pickle("../PickleFiles/AVgrades.pkl")

rb_data = rb_data.rename(columns={'recent_team': 'team'})
wrte_data = wrte_data.rename(columns={'recent_team': 'team'})
qb_data = qb_data.rename(columns={'recent_team': 'team'})
AVgrades = AVgrades.rename(columns={'year': 'season'})

qb_data.columns


Index(['player_id', 'season', 'GP', 'player_name', 'player_display_name',
       'position', 'position_group', 'headshot_url', 'season_type', 'team',
       ...
       'fantasy_points', 'fantasy_points_ppr', 'age', 'status', 'comp %',
       'td:int', 'yards/attempts', 'yards/comp', 'yards/carry',
       'passer rating'],
      dtype='object', length=122)

In [ ]:
import pandas as pd

# Ensure the season column is correctly formatted for merging
qb_data['season'] = qb_data['season'].astype(int)  # Ensure the year is an integer

AVgrades['season'] = AVgrades['season'].astype(int)

# Teams use NFL abbreviations from nflverse, matching AVgrades

qb_data = pd.merge(qb_data, AVgrades[['team','season', 'oline', 'qb', 'rb', 'wrte', 'dst']], on=['team', 'season'], how='left')
qb_data = qb_data[['player_id', 'season', 'player_display_name', 'team',
       'GP', 'position', 'age', 'season_type', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks',
       'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs',
       'passing_2pt_conversions','carries', 'rushing_yards',
       'rushing_tds','rushing_fumbles_lost',
       'rushing_first_downs','rushing_2pt_conversions','fantasy_points', 'oline', 'qb', 'rb', 'wrte', 'dst']]
qb_data.to_pickle("../PickleFiles/final_qb_data.pkl")
qb_data.columns

Index(['player_id', 'season', 'player_display_name', 'team', 'GP', 'position',
       'age', 'season_type', 'completions', 'attempts', 'passing_yards',
       'passing_tds', 'interceptions', 'sacks', 'sack_fumbles_lost',
       'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs',
       'passing_2pt_conversions', 'carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles_lost', 'rushing_first_downs',
       'rushing_2pt_conversions', 'fantasy_points', 'oline', 'qb', 'rb',
       'wrte', 'dst'],
      dtype='object')

In [ ]:
import pandas as pd

# Ensure the season column is correctly formatted for merging
rb_data['season'] = rb_data['season'].astype(int)  # Ensure the year is an integer

AVgrades['season'] = AVgrades['season'].astype(int)

# Teams use NFL abbreviations from nflverse, matching AVgrades

rb_data = pd.merge(rb_data, AVgrades[['team','season', 'oline', 'qb', 'rb', 'wrte', 'dst']], on=['team', 'season'], how='left')
rb_data = rb_data[['player_id', 'season', 'player_display_name', 'team',
       'GP', 'position', 'age', 'season_type', 'carries', 'rushing_yards', 'rushing_tds',
                        'rushing_fumbles_lost', 'rushing_first_downs',
                        'rushing_2pt_conversions', 'receptions', 'targets',
                        'receiving_yards', 'receiving_tds',
                        'receiving_fumbles_lost', 'receiving_air_yards',
                        'receiving_yards_after_catch', 'receiving_first_downs',
                        'receiving_2pt_conversions','special_teams_tds',
                        'rrtd', 'fantasy_points', 'oline', 'qb', 'rb', 'wrte', 'dst']]
rb_data.to_pickle("../PickleFiles/final_rb_data.pkl")
rb_data.columns

Index(['player_id', 'season', 'player_display_name', 'team', 'GP', 'position',
       'age', 'season_type', 'carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles_lost', 'rushing_first_downs',
       'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards',
       'receiving_tds', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs',
       'receiving_2pt_conversions', 'special_teams_tds', 'rrtd',
       'fantasy_points', 'oline', 'qb', 'rb', 'wrte', 'dst'],
      dtype='object')

In [ ]:
import pandas as pd

# Ensure the season column is correctly formatted for merging
wrte_data['season'] = wrte_data['season'].astype(int)  # Ensure the year is an integer

AVgrades['season'] = AVgrades['season'].astype(int)

# Teams use NFL abbreviations from nflverse, matching AVgrades

wrte_data = pd.merge(wrte_data, AVgrades[['team','season', 'oline', 'qb', 'rb', 'wrte', 'dst']], on=['team', 'season'], how='left')
wrte_data = wrte_data[['player_id', 'season', 'player_display_name', 'team',
       'GP', 'position', 'age', 'season_type', 'carries', 'rushing_yards', 'rushing_tds',
                        'rushing_fumbles_lost', 'rushing_first_downs',
                        'rushing_2pt_conversions', 'receptions', 'targets',
                        'receiving_yards', 'receiving_tds',
                        'receiving_fumbles_lost', 'receiving_air_yards',
                        'receiving_yards_after_catch', 'receiving_first_downs',
                        'receiving_2pt_conversions','special_teams_tds',
                        'rrtd', 'fantasy_points', 'oline', 'qb', 'rb', 'wrte', 'dst']]
wrte_data.to_pickle("../PickleFiles/final_wrte_data.pkl")
wrte_data.columns

Index(['player_id', 'season', 'player_display_name', 'team', 'GP', 'position',
       'age', 'season_type', 'carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles_lost', 'rushing_first_downs',
       'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards',
       'receiving_tds', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs',
       'receiving_2pt_conversions', 'special_teams_tds', 'rrtd',
       'fantasy_points', 'oline', 'qb', 'rb', 'wrte', 'dst'],
      dtype='object')